# Purdon et al. (2013) on dandiset 001285

**Paper.** Purdon PL, Pierce ET, Mukamel EA, et al. (2013). Electroencephalogram signatures of loss
and recovery of consciousness from propofol. *PNAS* 110(12):E1142-E1151.
[doi:10.1073/pnas.1221180110](https://doi.org/10.1073/pnas.1221180110). The claims file was written
from the version of record in PubMed Central (PMC3607036), with the Supporting Information.

**Dandiset.** [DANDI:001285](https://dandiarchive.org/dandiset/001285/0.250110.2125), version
0.250110.2125 (doi:10.48324/dandi.001285/0.250110.2125). The paper predates the dandiset and cites
no version, so the most recent published version is pinned. It is also the only published version;
the cell below shows that the draft holds the same single file (the same blob).

**What the dandiset holds.** One NWB file, one subject, with one generic `TimeSeries` of 64 EEG
channels at 250 Hz over 154 min. It holds no channel names or electrode positions, no behavioural
task (stimuli and button presses), no infusion levels or times, and no behavioural markers (EON,
LOC, ROC, EOFF). The paper studied 10 subjects.

**Claims.** All 22 claims of the claims file are blocked. Every one needs data the dandiset does
not hold: the behavioural responses (from which every time marker the EEG analyses are aligned to
is derived), the other nine subjects (the group statistics, sign tests across subjects and
subject counts), or the channel identities (the frontal channel near Fz, the Laplacian, the
spatial groups). The outcome of every claim is blocked with data missing as the reason; for fifteen
of the sixteen EEG claims the missing channel names and positions are a second reason (metadata
insufficient), and the file has no infusion times either. This notebook shows what the file holds
and checks its description against the values; it tests no claim.

| Claim | Needs | Outcome (second reason) |
|---|---|---|
| fig1c-clicks-lost-before-verbal | behaviour, 10 subjects | blocked: data missing |
| eon-loc-duration | behaviour, 10 subjects | blocked: data missing |
| fig1e-group-induction | behaviour, 10 subjects | blocked: data missing |
| fig1e-group-emergence | behaviour, 10 subjects | blocked: data missing |
| eoff-not-reached | behaviour, 10 subjects | blocked: data missing |
| roc-eoff-duration | behaviour, 10 subjects | blocked: data missing |
| fig2bc-gamma-beta-before-loc | LOC, 10 subjects, Fz | blocked: data missing (metadata insufficient) |
| fig2bc-alpha-slow-at-loc | LOC, 10 subjects, Fz | blocked: data missing (metadata insufficient) |
| fig2d-spatial-loc15 | LOC, 10 subjects, channel labels | blocked: data missing (metadata insufficient) |
| fig2-emergence-reverse | ROC, EOFF, 10 subjects, Fz | blocked: data missing (metadata insufficient) |
| slow-power-tenfold | LOC/ROC, 10 subjects, Fz | blocked: data missing (metadata insufficient) |
| fig3a-induction-traveling-peak | EON, LOC, 10 subjects, Fz | blocked: data missing (metadata insufficient) |
| fig3a-emergence-traveling-peak | ROC, EOFF, 10 subjects, Fz | blocked: data missing (metadata insufficient) |
| fig3b-frontal-median-frequency-power | all markers, 10 subjects, channel labels | blocked: data missing (metadata insufficient) |
| figs2-band-spatial-shift | all markers, 10 subjects, channel labels | blocked: data missing (metadata insufficient) |
| fig4ab-occipital-alpha-mode | LOC/ROC, 10 subjects, channel labels | blocked: data missing (metadata insufficient) |
| fig4cd-frontal-alpha-mode | LOC/ROC, 10 subjects, channel labels | blocked: data missing (metadata insufficient) |
| fig4-no-low-frequency-mode | LOC/ROC, 10 subjects | blocked: data missing |
| fig5-trough-max-induction | LOC, 10 subjects, Fz | blocked: data missing (metadata insufficient) |
| fig5-peak-max | LOC, 10 subjects, Fz | blocked: data missing (metadata insufficient) |
| fig5-trough-max-emergence | ROC, 10 subjects, Fz | blocked: data missing (metadata insufficient) |
| figs3-broadband-modulation | LOC, 10 subjects, Fz | blocked: data missing (metadata insufficient) |

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pynwb
import requests

import propofol_eeg as pe

np.set_printoptions(linewidth=150, suppress=True)
started = time.time()
meta = pe.version_metadata()
versions = pe.list_versions()
assets = pe.list_assets()
print(f"dandiset {pe.DANDISET} version {pe.VERSION}, doi {meta.get('doi')}")
print("versions:", [(v["version"], v["asset_count"], v["size"]) for v in versions])
draft_assets = requests.get(f"{pe.API}/dandisets/{pe.DANDISET}/versions/draft/assets/", timeout=60).json()["results"]
print("published assets:", [(a["path"], a["size"], a["blob"]) for a in assets])
print("draft assets:    ", [(a["path"], a["size"], a["blob"]) for a in draft_assets])
print("draft holds the same blobs:", sorted(a["blob"] for a in assets) == sorted(a["blob"] for a in draft_assets))
print("\nassetsSummary:", meta["assetsSummary"])
print("\ndescription:", meta["description"])

## Data check

Everything the file holds, then the values. Each check of the description against the values is
stated after the output.

In [ ]:
f = pe.open_asset(assets[0]["asset_id"])
print("\n".join(pe.describe_hdf5(f)))
io = pynwb.NWBHDF5IO(file=f, load_namespaces=True)
nwb = io.read()
ts = nwb.acquisition["example_timeseries"]
print("\npynwb reads:", type(ts).__name__, "| rate", ts.rate, "Hz | starting_time", ts.starting_time, "s | unit", ts.unit,
      "| data shape", ts.data.shape, "| num_samples", ts.num_samples)
print("subject:", nwb.subject.subject_id, "| age", nwb.subject.age, "| sex", nwb.subject.sex, "| species", nwb.subject.species)
print("session_start_time:", nwb.session_start_time, "| file_create_date:", nwb.file_create_date)
print("identifier:", nwb.identifier, "| session_description:", nwb.session_description)
print("notes:", nwb.notes)
print("contributors on DANDI:", [(c["name"], c.get("identifier")) for c in meta["contributor"]])
print("\nelectrodes table:", nwb.electrodes, "| electrode groups:", dict(nwb.electrode_groups), "| devices:", dict(nwb.devices))
print("trials:", nwb.trials, "| epochs:", nwb.epochs, "| intervals:", dict(nwb.intervals))
print("groups and datasets under general/:", sorted(f["general"].keys()))
print("general/extracellular_ephys present:", "general/extracellular_ephys" in f,
      "| contents:", sorted(f["general/extracellular_ephys"].keys()) if "general/extracellular_ephys" in f else None)
print("processing modules:", dict(nwb.processing), "| stimulus:", dict(nwb.stimulus), "| analysis:", dict(nwb.analysis))

The file is one `TimeSeries` named `example_timeseries`, its description "no description", with
data of shape 64 x 2,316,950 stored channels first. NWB stores time on the first axis, so pynwb
reads the series as 64 samples (`num_samples` above); the 64 rows are channels, as the description's
64-channel EEG and the 250 Hz rate over the second axis (154 min) imply. There is no electrode
table (`general/extracellular_ephys` is an empty group, and `general/devices` is empty too), so no
channel has a name, a position or a bad-channel mark. There are no trials, epochs,
intervals, events, stimuli or processing modules: the behavioural task, the infusion levels and
the time markers are absent. The subject's age is P30D (30 days), where the paper's subjects were
18 to 36 years old. The session start time equals the first file creation date (2025-01-10), the
day of the conversion, not of a recording made before 2013. The identifier (`NWB123`) and session
description ("EEG Session Example") are placeholders, and the ORCID in the notes differs from the
contact's ORCID on DANDI.

In [ ]:
data = f[pe.SERIES + "/data"]
fs = float(f[pe.SERIES + "/starting_time"].attrs["rate"])
n_ch, n_s = data.shape
print(f"channels {n_ch}, samples {n_s}, rate {fs} Hz, duration {n_s / fs / 60:.1f} min, unit {data.attrs['unit']!r}")
t0 = time.time()
summ = pe.channel_summaries(data)
print(f"read every channel in {time.time() - t0:.0f} s")
for k in ("mean", "sd", "min", "max"):
    v = summ[k]
    print(f"{k:>5} per channel (uV): min {v.min():9.1f}, median {np.median(v):9.1f}, max {v.max():9.1f}")
print("NaN samples in all channels:", int(summ["nan"].sum()))
print("longest run of identical consecutive samples, any channel:", int(summ["longest_flat"].max()))
print("distinct channels (by hash of all samples):", len(set(summ["hash"])), "of", n_ch)
cc = np.corrcoef(summ["decimated"])
iu = np.triu_indices(n_ch, 1)
order = np.argsort(-cc[iu])
print("highest correlations between channels (rows 0-63, below 4 Hz):",
      [(int(iu[0][j]), int(iu[1][j]), round(float(cc[iu][j]), 3)) for j in order[:5]])
print("channels with SD above 300 uV (rows):", np.flatnonzero(summ["sd"] > 300).tolist())

No channel is a copy of another (64 distinct hashes, highest correlation 0.83 below 4 Hz), so no
recording is deposited twice. There are no missing samples and no flat stretches (no two
consecutive samples are equal in any channel). The values are consistent with microvolts (the
spectrum below has eyes-closed alpha of a few tens of uV^2/Hz), but the SDs and extremes are large,
and several channels have SDs of several hundred uV, which suggests noisy channels the paper would
have excluded by inspection; the file does not mark them.

In [ ]:
freqs, psd = pe.channel_spectra(data, fs)
med = np.median(psd, axis=0)
def at(hz):
    return float(med[np.argmin(abs(freqs - hz))])
print("median spectrum over channels (uV^2/Hz):")
for hz in (1, 10, 20, 40, 55, 59, 60, 61, 65, 100, 115, 120, 124):
    print(f"  {hz:5.1f} Hz  {at(hz):.3g}")
above = freqs > 40
print("largest peak above 40 Hz at", float(freqs[above][np.argmax(med[above])]), "Hz")
print(f"power at 120 Hz over power at 100 Hz: {at(120) / at(100):.2e}")

The largest peak above 40 Hz is at 60 Hz, the mains frequency in the United States, where the
paper's data were recorded; with a wrong sampling rate it would sit elsewhere. Power falls by more
than three orders of magnitude between 100 and 120 Hz, the anti-alias filter the Methods describe
before down-sampling to 250 Hz (Methods, EEG Preprocessing). The stated rate of 250 Hz is
consistent with the data.

The paper's protocol (Methods, Experimental Design) holds levels of 14 min (0, 1, 2, 3, 4 and 5
ug/mL, then four emergence levels), about 140 min in all; the archive's description adds a 20-min
baseline and a 20-min recording after the infusion. The file holds 154.5 min with no marker of any
level. The band power over time, below, is shown to see whether the recording has the structure of
such a session. It is a description of the file, not a test of a claim: without LOC, ROC, the
channel identities and the other subjects, none of the claims' quantities can be computed.

In [ ]:
starts, bp = pe.block_band_power(data, fs)
ref = slice(0, 15)       # the first 15 one-minute blocks, chosen after looking at the data: before the first change seen below
quiet = summ["sd"] <= 300   # variant leaving out the 9 channels with SD above 300 uV
fig, axes = plt.subplots(len(bp), 1, figsize=(9, 8), sharex=True)
for ax, (band, v) in zip(axes, bp.items()):
    ratio = np.median(v / v[:, ref].mean(axis=1, keepdims=True), axis=0)
    ax.semilogy(starts / 60, ratio, color="#3a5a8c", lw=1.5)
    ax.axhline(1, color="0.6", lw=0.8)
    ax.set_ylabel(band + "\nover min 0-15", fontsize=8)
    ax.grid(axis="y", color="0.9")
    print(f"{band}: ratio to minutes 0-15, median over 64 channels; range {ratio.min():.2f} to {ratio.max():.1f}")
    for lo, hi in ((0, 15), (16, 32), (33, 47), (48, 60), (60, 100), (100, 106), (110, 154)):
        print(f"   minutes {lo:3d}-{hi:3d}: median {np.median(ratio[lo:hi]):6.2f}")
    print("   per minute:", np.round(ratio, 1).tolist())
    ratio_q = np.median(v[quiet] / v[quiet][:, ref].mean(axis=1, keepdims=True), axis=0)
    print(f"   without the {int((~quiet).sum())} channels with SD above 300 uV, median over {int(quiet.sum())}:",
          "; ".join(f"min {lo}-{hi} {np.median(ratio_q[lo:hi]):.2f}" for lo, hi in ((16, 32), (33, 47), (48, 60), (60, 100), (110, 154))))
axes[-1].set_xlabel("time from start of file (min)")
fig.suptitle("Band power per 1-min block over its mean in minutes 0-15, median over the 64 channels", fontsize=10)
fig.tight_layout()
plt.show()
print("all points are within the axes")

The reference period (minutes 0 to 15) was chosen after looking at the data, and the medians are
over all 64 channels without a Laplacian. The time course is consistent in outline with a propofol
session of the paper's design. Beta power steps up
at minute 17 (about 2 times the first 15 min) and again at minute 33 (6 to 9 times), about one 14-min
level apart. Alpha and low-frequency power rise together at minute 48; from minute 49 to minute 97
low-frequency power is 6.5 to 34 times the first 15 min (median 15.4 over minutes 60 to 100) and
alpha 2.5 to 8.2 times. Both are back near their opening levels from minute 105, while from minute
105 to 120 beta stays at 1.2 to 1.7 times and gamma at 1.1 to 1.7 times. Single minutes with very large
low-frequency ratios (25 at minute 33, 88 at minute 46, 347 at minute 48) look like artifacts; they
are kept, and enter only these descriptive medians. Leaving out the 9 channels with SDs above 300 uV
changes little: the low-frequency median over minutes 60 to 100 is 16.3 over the other 55 channels,
beta over minutes 33 to 47 is 8.5, and alpha over minutes 48 to 60 is 7.3 (4.7 over all 64).
This supports the description only in outline, and is no evidence for slow-power-tenfold or the
Figure 2 claims: the reference was chosen by eye, the unconscious period is not marked, and the
channels are not the paper's frontal channel. Which minute is LOC or ROC cannot be read from the
file, as the paper defines them from the behavioural responses, which are not deposited.

**Clock.** There are no trial tables, events or intervals in the file, so no stream can be shown
to share a clock with a trial table; this could not be established for the one session.

## What each claim needs, and what the file lacks

No claim is tested. The cell below checks, from the file and the archive, each kind of data the
claims rest on.

In [ ]:
others = []
search_terms = ("propofol", "Purdon", "Yousefi", "anesthesia", "consciousness", "Emery Brown", "EEG")
for q in search_terms:
    for r in requests.get(f"{pe.API}/dandisets/", params={"search": q, "page_size": 50}, timeout=60).json()["results"]:
        if r["identifier"] != pe.DANDISET and r["identifier"] not in [o[0] for o in others]:
            d = requests.get(f"{pe.API}/dandisets/{r['identifier']}/versions/draft/", timeout=60).json()
            others.append((r["identifier"], (r.get("draft_version") or {}).get("name"),
                           "Purdon" in d.get("description", ""), d["assetsSummary"].get("numberOfFiles")))
print(f"other dandisets found by searching {search_terms}: {len(others)}")
print("of these, those whose description names Purdon (id, name, files):")
for o in others:
    if o[2]:
        print("  ", o[0], o[1], o[3])
needs = {
    "subjects in the dandiset (paper: 10)": meta["assetsSummary"]["numberOfSubjects"],
    "behavioural responses or stimulus times": any(k in f for k in ("stimulus/presentation", "intervals/trials")) and len(f["stimulus/presentation"]) > 0,
    "infusion levels or behavioural markers (EON, LOC, ROC, EOFF)": len(f["intervals"]) > 0 or len(f["processing"]) > 0,
    "channel names, positions or bad-channel marks (an electrode table)": "general/extracellular_ephys/electrodes" in f,
}
for k, v in needs.items():
    print(f"{k}: {v}")
print(f"\nnotebook ran in {(time.time() - started) / 60:.1f} min")

Searching DANDI for seven terms (propofol, Purdon, Yousefi, anesthesia, consciousness, Emery Brown,
EEG) found 31 other dandisets. Two of them, 001286 and 001287, carry the same description of Purdon
et al. (2013) and hold no files. No behavioural data, markers or channel identities were found for this subject, and
no data for the other nine.

## Summary

| Claim | Outcome |
|---|---|
| All 22 claims (Figures 1 to 5, S2, S3) | blocked: data missing |
| of these, 15 EEG claims that need channel identities | second reason: metadata insufficient |

Every claim is blocked. The six behavioural claims need the button-press responses to the
auditory task, which are not in the dandiset. The sixteen EEG claims are aligned to LOC, ROC, EON
or EOFF, which the paper derives from those responses, rest on the median or a sign test across
10 subjects, of which the dandiset holds one, and, for most, on channel identities (a frontal
channel near Fz, the Laplacian's neighbours, frontal and occipital groups), which the file does
not give.

What was found about the dataset:

- One of the paper's ten subjects is deposited; dandisets 001286 and 001287, with the same
  description, are empty.
- No behaviour, infusion levels or time markers are deposited.
- The 64 channels have no names or positions, and noisy channels are not marked.
- The EEG is stored channels first in a generic `TimeSeries`, so NWB tools read it as 64 samples.
- Subject and session metadata are placeholders or wrong: age 30 days for an adult volunteer, a
  session start time equal to the conversion date, identifier `NWB123`, session description "EEG
  Session Example".
- The values themselves are consistent with the description: 250 Hz (60 Hz mains peak, anti-alias
  roll-off), microvolts, no duplicated channels or gaps, and a band-power time course with the
  outline of a gradual propofol induction and emergence, against a reference period chosen by eye.